## Анализ результатов A/B-тестирования интернет-магазина BitMotion Kit

Интернет-магазин BitMotion Kit продаёт геймифицированные товары для людей, ведущих здоровый образ жизни. По отзывам пользователей, текущий интерфейс сайта оказался слишком сложным, поэтому компания разработала новую версию сайта и протестировала её на части пользователей.

Гипотеза теста: упрощение интерфейса должно увеличить конверсию зарегистрированных пользователей в покупателей в течение семи дней после регистрации минимум на 3 процентных пункта.

В рамках проекта проведена проверка корректности A/B-теста и анализ пользовательских действий в группах A и B.

# Цель и задачи проекта.



**Цель проекта**

Оценить влияние упрощения интерфейса интернет-магазина на конверсию зарегистрированных пользователей в покупателей в течение первых семи дней после регистрации.

**Задачи**

- Загрузить данные об участниках A/B-теста и пользовательских событиях.
- Отобрать пользователей, участвующих в тесте interface_eu_test, и проверить: наличие только групп A и B, а также равномерность распределение пользователей по группам.
- Ограничить горизонт анализа первыми семью днями с момента регистрации пользователя.
- Проверить достаточность выборки для обнаружения заданного эффекта.
- Рассчитать для каждой группы: общее количество пользователей, конверсию в покупку.
- Проверить гипотезу о росте конверсии в тестовой группе с помощью подходящего статистического теста (z-тест для двух пропорций).
- Сформулировать вывод о том, оказало ли интерфейсное изменение статистически значимое влияние на пользовательское поведение.


##  Загружаем данные для оценки их целостность.


In [1]:
import pandas as pd 

import numpy as np

from scipy.stats import ttest_ind

from statsmodels.stats.proportion import proportions_ztest

from statsmodels.stats.power import NormalIndPower


In [2]:
participants = pd.read_csv('/datasets/ab_test_participants.csv')
events = pd.read_csv('/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [3]:
print(participants.info())
display(participants.head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB
None


,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac
5,002412F1EB3F6E38,B,interface_eu_test,Mac
6,002540BE89C930FB,B,interface_eu_test,Android
7,0031F1B5E9FBF708,A,interface_eu_test,Android
8,003346BB64227D0C,B,interface_eu_test,Android
9,00341D8401F0F665,A,recommender_system_test,iPhone


In [4]:
# Проверяем данные на явные дубликаты
participants_dupl = participants.duplicated(subset=['user_id','ab_test','device']).sum()

print(participants_dupl)

0


In [5]:
print(events.info())
display(events.head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB
None


,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN
5,AA346F4D22148024,2020-12-01 00:01:46,registration,-2.0
6,7EF01D0E72AF449D,2020-12-01 00:02:06,registration,-5.0
7,9A6276AD14B14252,2020-12-01 00:02:20,registration,-2.0
8,9B186A3B1A995D36,2020-12-01 00:02:37,registration,-3.5
9,9A6276AD14B14252,2020-12-01 00:02:53,login,NaN


In [6]:
# Проверяем данные на дублиуаты по набору столбцов
events_dupl = events.duplicated(subset=['user_id','event_dt','event_name','details' ]).sum()
print(events_dupl)

36318


In [7]:
# Выводим дубликаты
show_events = events[
    events.duplicated(
        subset=['user_id', 'event_dt', 'event_name', 'details'],
        keep=False
    )
].sort_values(
    by=['user_id', 'event_dt']
)

display(show_events.head(5))


,user_id,event_dt,event_name,details
309139,000199F1887AE5E6,2020-12-14 09:57:42,login,NaN
309140,000199F1887AE5E6,2020-12-14 09:57:42,login,NaN
346516,000199F1887AE5E6,2020-12-15 07:28:44,product_page,NaN
346517,000199F1887AE5E6,2020-12-15 07:28:44,product_page,NaN
346519,000199F1887AE5E6,2020-12-15 07:28:44,product_page,NaN


In [8]:
# Удаляем дубликаты
events_clean = events.drop_duplicates(
    subset=['user_id', 'event_dt', 'event_name', 'details']
).reset_index(drop=True)


#Проверяем рузультат 
print(events_clean.duplicated(subset=['user_id', 'event_dt', 'event_name', 'details']).sum())



0


В ходе знакомства с данными, в таблице events было выявлено 36318 полных дубликатов. Для дальнейшего анализа дублирующие строки были очищены. 

## Оцениваем корректность проведения теста по таблице `ab_test_participants`:

   3\.1 Выделяем пользователей, участвующих в тесте
   
   Проверяем:
  

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [9]:
# Оставляем только нужный тест interface_eu_test 
test_name = 'interface_eu_test'
test_part = participants[participants['ab_test'] == test_name].copy()

print('Всего пользователей в файле:', participants['user_id'].nunique())
print(f'Участников теста {test_name}:', test_part['user_id'].nunique()) 
print('\nРаспределение по тестам:')
print(participants['ab_test'].value_counts())


Всего пользователей в файле: 13638
Участников теста interface_eu_test: 10850

Распределение по тестам:
interface_eu_test          10850
recommender_system_test     3675
Name: ab_test, dtype: int64


In [10]:
# Проверяем, что в нужном тесте только группы A и B 
print('\nГруппы в тесте interface_eu_test:')
print(test_part['group'].value_counts(dropna=False))



Группы в тесте interface_eu_test:
B    5467
A    5383
Name: group, dtype: int64


In [11]:
# Равномерность распределения по группам 
group_sizes = (
    test_part
    .groupby('group')['user_id']
    .nunique()
    .rename('users')
    .reset_index()
)

total_users_test = group_sizes['users'].sum() 
group_sizes['share'] = group_sizes['users'] / total_users_test*100

print('\nРазмер и доля пользователей по группам:')
display(group_sizes)



Размер и доля пользователей по группам:


,group,users,share
0,A,5383,49.612903
1,B,5467,50.387097


In [12]:
# Поиск пересечений с другими тестами 
multi_test_users = (
    participants
    .groupby('user_id')['ab_test']
    .nunique()
)
multi_test_users = multi_test_users[multi_test_users > 1].index

print('\nПользователей, участвующих более чем в одном тесте:', len(multi_test_users))

# пересечения именно с нашим тестом interface_eu_test 
intersection_users = pd.Series(
    test_part['user_id'].unique()
).isin(multi_test_users).sum()

print('Из них пересекаются с interface_eu_test:', intersection_users)



Пользователей, участвующих более чем в одном тесте: 887
Из них пересекаются с interface_eu_test: 887


In [13]:
# Проверка: пользователь в нескольких группах одного теста 
group_overlap = (participants.groupby(['user_id','ab_test'])['group'].nunique())

group_overlap = group_overlap[group_overlap > 1]

print('Пользователей, попавших более чем в одну группу теста:', len(group_overlap))


Пользователей, попавших более чем в одну группу теста: 0


Пересечение групп между сосбой не выявлено

In [14]:
events_clean = test_part[~test_part['user_id'].isin(multi_test_users)].copy()
print('\nПосле удаления пересечений пользователей в тесте осталось:',events_clean['user_id'].nunique())



После удаления пересечений пользователей в тесте осталось: 9963


3\.2 Проанализируем данные о пользовательской активности по таблице `ab_test_events`:

- оставим только события, связанные с участвующими в изучаемом тесте пользователями;

In [15]:
# Объединяем события с участниками теста
events_test = events.merge(
    participants[['user_id', 'group', 'ab_test']],
    on='user_id',
    how='inner'
)

# Проверяем размеры
print(events.shape, events_test.shape)

# Проверяем, что других пользователей больше нет
print(events_test['user_id'].isin(participants['user_id']).all())


(787286, 4) (104558, 6)
True


Обновлена таблица events_test, теперь она содержит только события пользователей теста, по ммимо этого добавлены группа (group) и название теста (ab_test).


Определяем горизонт анализа: расчитываем время (лайфтайм) совершения события пользователем после регистрации и оставим только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [16]:
# Дата регистрации пользователя (минимальная дата события пользователя)
registration_dates = (
    events_test.groupby('user_id', as_index=False)['event_dt'].min().rename(columns={'event_dt': 'registration_dt'}))


# Добавим дату регистрации к событиям
events_test = events_test.merge(
    registration_dates,
    on='user_id',
    how='left'
)

display(events_test.head(5))


,user_id,event_dt,event_name,details,group,ab_test,registration_dt
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0,A,interface_eu_test,2020-12-06 14:10:01
1,5F506CEBEDC05D30,2020-12-07 01:25:14,login,NaN,A,interface_eu_test,2020-12-06 14:10:01
2,5F506CEBEDC05D30,2020-12-07 01:25:47,login,NaN,A,interface_eu_test,2020-12-06 14:10:01
3,5F506CEBEDC05D30,2020-12-09 12:40:49,login,NaN,A,interface_eu_test,2020-12-06 14:10:01
4,5F506CEBEDC05D30,2020-12-09 12:40:49,product_page,NaN,A,interface_eu_test,2020-12-06 14:10:01


In [17]:
#Рассчитаем лайфтайм события
events_test['lifetime_days'] = (events_test['event_dt'] - events_test['registration_dt']).dt.days

display(events_test.head(5))

,user_id,event_dt,event_name,details,group,ab_test,registration_dt,lifetime_days
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0,A,interface_eu_test,2020-12-06 14:10:01,0
1,5F506CEBEDC05D30,2020-12-07 01:25:14,login,NaN,A,interface_eu_test,2020-12-06 14:10:01,0
2,5F506CEBEDC05D30,2020-12-07 01:25:47,login,NaN,A,interface_eu_test,2020-12-06 14:10:01,0
3,5F506CEBEDC05D30,2020-12-09 12:40:49,login,NaN,A,interface_eu_test,2020-12-06 14:10:01,2
4,5F506CEBEDC05D30,2020-12-09 12:40:49,product_page,NaN,A,interface_eu_test,2020-12-06 14:10:01,2


In [18]:
#Ограничим горизонт первыми 7 днями

events_7d = events_test[
    (events_test['lifetime_days'] >= 0) &
    (events_test['lifetime_days'] < 7)]

# Проверка результата 
display(events_7d.head(5))
print(events_7d['lifetime_days'].describe())



,user_id,event_dt,event_name,details,group,ab_test,registration_dt,lifetime_days
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0,A,interface_eu_test,2020-12-06 14:10:01,0
1,5F506CEBEDC05D30,2020-12-07 01:25:14,login,NaN,A,interface_eu_test,2020-12-06 14:10:01,0
2,5F506CEBEDC05D30,2020-12-07 01:25:47,login,NaN,A,interface_eu_test,2020-12-06 14:10:01,0
3,5F506CEBEDC05D30,2020-12-09 12:40:49,login,NaN,A,interface_eu_test,2020-12-06 14:10:01,2
4,5F506CEBEDC05D30,2020-12-09 12:40:49,product_page,NaN,A,interface_eu_test,2020-12-06 14:10:01,2


count    90698.000000
mean         1.107070
std          1.710587
min          0.000000
25%          0.000000
50%          0.000000
75%          2.000000
max          6.000000
Name: lifetime_days, dtype: float64


Оцениваем достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [19]:
# Заданные параметры
baseline_conversion = 0.30     # базовая конверсия
mde = 0.05                     # минимально обнаружимый эффект (+5 п.п.)
alpha = 0.05                   # уровень значимости (1 - 95%)
power = 0.80                   # мощность теста

# Конверсия в тестовой группе
conversion_test = baseline_conversion + mde

# Размер эффекта для пропорций
effect_size = 2 * (
    np.arcsin(np.sqrt(conversion_test)) -
    np.arcsin(np.sqrt(baseline_conversion))
)

# Расчёт размера выборки
analysis = NormalIndPower()

sample_size_per_group = analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1
)

print(f'Минимальный размер выборки на группу: {np.ceil(sample_size_per_group)}')
print(f'Минимальный общий размер выборки: {np.ceil(sample_size_per_group) * 2}')



Минимальный размер выборки на группу: 1376.0
Минимальный общий размер выборки: 2752.0


**Интерпретация результата**

- Минимально необходимое количество пользователей в каждой группе - 1376
- Фактическая выборка больше — тесту достаточно мощности, вероятность обнаружить эффект соответствует заданным 80%


Для каждой группы рассчитываем количество посетителей, сделавших покупку, и общее количество посетителей.

In [20]:
# Оставляем только события за 7 дней очищенных пользователей 
events_test_7d = events_7d[
    events_7d['user_id'].isin(events_clean['user_id'])
].copy()

# Покупатели
buyers = (
    events_test_7d[events_test_7d['event_name'] == 'purchase']
    [['user_id', 'group']]
    .drop_duplicates()
)

# Общее число пользователей
total_users = (
    events_clean
    .groupby('group')['user_id']
    .nunique()
    .reset_index(name='total_users')
)

# Покупатели по группам
buyers_count = (
    buyers
    .groupby('group')['user_id']
    .nunique()
    .reset_index(name='buyers')
)

# Итог
result = total_users.merge(
    buyers_count,
    on='group',
    how='left'
).fillna({'buyers': 0})

display(result)


,group,total_users,buyers
0,A,4952,1377
1,B,5011,1480


In [21]:
# Конверсия

result['conversion'] = result['buyers'] / result['total_users'] 

result['conversion_pct'] = result['conversion'] * 100

display(result)


,group,total_users,buyers,conversion,conversion_pct
0,A,4952,1377,0.278069,27.806947
1,B,5011,1480,0.295350,29.535023


In [22]:
# Проверим,что все пользователи events_7d есть в events_clean
display(events_test_7d['user_id'].isin(events_clean['user_id']).all())

True

Сделаем предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.

**Предварительный вывод об изменении пользовательской активности в тестовой группе**

В A/B-тесте interface_eu_test после очистки данных от пользователей, участвующих более чем в одном тесте, осталось 9963 пользователя, распределённых между группами A и B достаточно равномерно. 
Минимально необходимый размер выборки для получения статистически значимых результатов  - 1376 пользователей на группу. Размер существующей выборки превышен, следовательно, выборка является достаточной с точки зрения мощности теста.

Анализ пользовательской активности в течение первых 7 дней с момента регистрации показал следующее:
- В контрольной группе (4952) пользователей меньше, чем в тестовой(5011);
- Количество пользователей, совершивших покупку, также выше в группе В (1480), чем в А(1377).
- Относительное значение конверсии в покупку в группе A составляет 27,8%, в группе B — 29,5%.

Таким образом, в тестовой группе наблюдается более высокая пользовательская активность, выраженная в большей доле пользователей, совершивших покупку, по сравнению с контрольной группой. Это может указывать на положительное влияние тестируемого интерфейсного изменения на поведение пользователей.

Однако данный вывод носит предварительный характер. Для подтверждения того, что наблюдаемое различие не является случайным, необходимо провести статистическую проверку гипотезы и оценить уровень статистической значимости полученного эффекта.



## Проводим оценку результатов A/B-тестирования:

Проверим изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

Будем сравнивать конверсии в двух независимых группах, для этого подойдет z-тест для двух пропорций, так как:
- анализируемая метрика — бинарная (покупка / отсутствие покупки);
- группы независимы;
- объёмы выборок достаточно велики;
- требуется сравнить доли пользователей, совершивших покупку.


Нулевая гипотеза (Н0): Конверсия в тестовой группе не отличается от конверсии в контрольной группе
- Н0 : рА = рВ

Альтернативная гипотеза(Н1): Конверсия в тестовой группе выше, чем в контрольной.
- Н1 : рВ > рA

Используем однасторонний тип проверки и уровень значимости: а = 0,05

In [23]:
# Данные (Сначала В потом А)
buyers = np.array([1480, 1377])
total_users = np.array([5011, 4952])

# Односторонний z-тест (B > A)
stat, p_value = proportions_ztest(
    count=buyers,
    nobs=total_users,
    alternative='larger'
)

print(stat, p_value)



1.9069651971471773 0.028262547212292124


In [24]:
#Принятие статистического решения

alpha = 0.05

if p_value < alpha:
    print('Отвергаем H₀: конверсия в тестовой группе статистически значимо выше')
else:
    print('Нет оснований отвергать H₀: статистически значимых различий не выявлено')


Отвергаем H₀: конверсия в тестовой группе статистически значимо выше


**Выводы по результатам A/B-тестирования**

Гипотезы

- Нулевая гипотеза (H₀): конверсия в тестовой группе B не выше, чем в контрольной группе A
- Альтернативная гипотеза (H₁): конверсия в тестовой группе B выше, чем в контрольной группе A

Результаты статистического теста
- Тип теста: односторонний z-тест для двух пропорций
Уровень значимости установлен α = 0.05

**Результаты теста:**
- Z-статистика: 1.97
- p-value: 0.028

Поскольку p-value < а, нулевая гипотеза отвергается.

Ожидаемый эффект был достигнут частично
Направление эффекта соответствует ожиданиям (конверсия выросла)и эффект статистически значим. Но величина роста ниже заявленного целевого значения.
Следовательно, интерфейсное изменение работает, но его влияние оказалось менее сильным, чем предполагалось на этапе планирования теста.


**Интерпретация результатов** 

Полученные результаты свидетельствуют о том, что конверсия в тестовой группе статистически значимо выше, чем в контрольной группе. Вероятность того, что наблюдаемое различие в доле покупателей между группами возникло случайно, составляет менее 5%.
Таким образом, данные подтверждают гипотезу о том, что изменение интерфейса положительно повлияло на конверсию пользователей в течение первых семи дней после регистрации.
При этом величина эффекта является умеренной, фактический рост конверсии ниже ожидаемого эффекта.


**Возможные причины полученного результата** 

1.	Недостаточная выраженность изменений интерфейса, новый дизайн мог улучшить пользовательский опыт, но не настолько, чтобы привести к более сильному росту конверсии.
2.	Влияние внешних факторов, решение о покупке может в большей степени зависеть от ассортимента, цены или мотивации пользователя, чем от интерфейсных изменений.


